## Download the data

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Data Preparation and Cleaning
* Look at some information about the data and columns
* Fix any missing or incorrect values

In [6]:
campaigns_df = pd.read_csv('data/campaigns.csv')
campaigns_df.head()

,id,campaign_type,channel,topic,started_at,finished_at,total_count,ab_test,warmup_mode,hour_limit,subject_length,subject_with_personalization,subject_with_deadline,subject_with_emoji,subject_with_bonuses,subject_with_discount,subject_with_saleout,is_test,position
0,63,bulk,mobile_push,sale out,2021-04-30 07:22:36.615023,2021-04-30 07:23:41,48211.0,NaN,False,NaN,146.0,False,False,True,False,False,False,NaN,NaN
1,64,bulk,mobile_push,sale out,2021-04-30 09:02:50.817227,2021-04-30 09:04:08,1037337.0,NaN,False,NaN,97.0,False,False,True,False,False,False,NaN,NaN
2,78,bulk,mobile_push,sale out,2021-05-06 07:14:10.533318,2021-05-06 07:15:17,70080.0,NaN,False,NaN,146.0,False,False,True,False,False,False,NaN,NaN
3,79,bulk,mobile_push,sale out,2021-05-06 09:03:56.486750,2021-05-06 09:42:15,921838.0,NaN,False,NaN,97.0,False,False,True,False,False,False,NaN,NaN
4,89,bulk,mobile_push,NaN,2021-05-07 11:54:06.168664,2021-05-07 11:54:38,45503.0,NaN,False,NaN,109.0,False,True,True,False,False,False,NaN,NaN


In [8]:
campaigns_df.isnull().sum()

id                                 0
campaign_type                      0
channel                            0
topic                             30
started_at                        83
finished_at                       99
total_count                       83
ab_test                         1895
warmup_mode                       77
hour_limit                      1876
subject_length                    27
subject_with_personalization      27
subject_with_deadline             27
subject_with_emoji                27
subject_with_bonuses              27
subject_with_discount             27
subject_with_saleout              27
is_test                         1880
position                        1883
dtype: int64

In [10]:
# percentage of missing values
# missing_percentages = campaigns_df.isna().sum().sort_values(ascending=False) / len(campaigns_df)
# missing_percentages

In [16]:
# missing_percentages[missing_percentages != 0].plot(kind='barh')

### Work with 'ab_test' column

In [19]:
campaigns_df['ab_test'] = campaigns_df['ab_test'].fillna(False).astype(bool)

C:\Users\User\AppData\Local\Temp\ipykernel_15668\1755339517.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  campaigns_df['ab_test'] = campaigns_df['ab_test'].fillna(False).astype(bool)


In [21]:
campaigns_df['is_test'] = campaigns_df.apply(
    lambda row: False if pd.isna(row['is_test']) and row['campaign_type'] in ['bulk', 'trigger'] else row['is_test'], axis=1
).astype(bool)

In [23]:
# for bulk и transactional company fill 0
campaigns_df.loc[campaigns_df['campaign_type'].isin(['bulk', 'transactional']), 'position'] = 0

trigger_median = campaigns_df.loc[campaigns_df['campaign_type'] == 'trigger', 'position'].median()
trigger_max = campaigns_df.loc[campaigns_df['campaign_type'] == 'trigger', 'position'].max()

campaigns_df.loc[
    (campaigns_df['campaign_type'] == 'trigger') & (campaigns_df['position'].isna()), 
    'position'
] = trigger_median if not pd.isna(trigger_median) else (trigger_max + 1 if not pd.isna(trigger_max) else 1)

campaigns_df['position'] = campaigns_df['position'].astype(int)

In [25]:
# For no bulk-company (trigger, transactional) fill 0
campaigns_df.loc[campaigns_df['campaign_type'] != 'bulk', 'hour_limit'] = 0

# find median for bulk-company with warmup_mode=True
warmup_median = campaigns_df.loc[
    (campaigns_df['campaign_type'] == 'bulk') & (campaigns_df['warmup_mode'] == True), 
    'hour_limit'
].median()

# fill NaN in bulk-company:
campaigns_df.loc[
    (campaigns_df['campaign_type'] == 'bulk') & (campaigns_df['warmup_mode'] == True) & (campaigns_df['hour_limit'].isna()),
    'hour_limit'
] = warmup_median if not pd.isna(warmup_median) else 1000  # Если медиана не найдена, используем 1000

campaigns_df.loc[
    (campaigns_df['campaign_type'] == 'bulk') & (campaigns_df['warmup_mode'] == False) & (campaigns_df['hour_limit'].isna()),
    'hour_limit'
] = 500  

campaigns_df['hour_limit'] = campaigns_df['hour_limit'].astype(int)


### Work with 'Topic' column
Filling missing values in the 'topic' column based on the mode (most frequent value)
within groups defined by 'campaign_type' and 'channel'

In [28]:
campaigns_df['topic'] = campaigns_df.groupby(['campaign_type', 'channel'])['topic'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

### Work with 'Total_count' column
Filling missing values in the 'total_count' column with the median value of the column.

In [31]:
campaigns_df['total_count'] = campaigns_df['total_count'].fillna(campaigns_df['total_count'].median())

### Work with 'Warmup_mode' column
Filling missing values in the 'warmup_mode' column with the most frequent value (mode).

In [34]:
campaigns_df['warmup_mode'] = campaigns_df['warmup_mode'].fillna(campaigns_df['warmup_mode'].mode()[0])
campaigns_df = campaigns_df.infer_objects(copy=False)

C:\Users\User\AppData\Local\Temp\ipykernel_15668\1786108946.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  campaigns_df['warmup_mode'] = campaigns_df['warmup_mode'].fillna(campaigns_df['warmup_mode'].mode()[0])


### Work with Subjects columns
'subject_length' = fill with 'median'
other subjects with  the most frequent value (mode).

In [37]:
campaigns_df['subject_length'] = campaigns_df['subject_length'].fillna(campaigns_df['subject_length'].median())

In [39]:
binary_cols = [
    'subject_with_personalization', 'subject_with_deadline', 'subject_with_emoji', 
    'subject_with_bonuses', 'subject_with_discount', 'subject_with_saleout'
]

for col in binary_cols:
    mode_value = campaigns_df[col].mode()[0]
    campaigns_df[col] = campaigns_df[col].fillna(mode_value)

campaigns_df = campaigns_df.infer_objects(copy=False)

C:\Users\User\AppData\Local\Temp\ipykernel_15668\2048126650.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  campaigns_df[col] = campaigns_df[col].fillna(mode_value)


### Work with 'start_at' and 'finished_at' columns

In [49]:
campaigns_df.isna().sum()

id                               0
campaign_type                    0
channel                          0
topic                            0
started_at                      83
finished_at                     99
total_count                      0
ab_test                          0
warmup_mode                      0
hour_limit                       0
subject_length                   0
subject_with_personalization     0
subject_with_deadline            0
subject_with_emoji               0
subject_with_bonuses             0
subject_with_discount            0
subject_with_saleout             0
is_test                          0
position                         0
dtype: int64